# ============================================================
# FINAL PIPELINE — ML1 Regression: annual.pay.usd
Model: SVR RBF | Selection: Statistical (~126 features)

Best params: C=0.5, epsilon=0.05, gamma=0.0005

Pipeline: VarianceThreshold -> Yeo-Johnson -> StandardScaler -> SVR
# ============================================================

In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PowerTransformer, StandardScaler, OneHotEncoder as OHE
from sklearn.feature_selection import VarianceThreshold, f_regression
from sklearn.impute import SimpleImputer
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# STEP 1 — LOAD RAW DATA
# ============================================================

train_raw = pd.read_csv('Data/train.csv')
test_raw  = pd.read_csv('Data/test.csv')

print(f"Train: {train_raw.shape}  |  Test: {test_raw.shape}")


In [ ]:
# ============================================================
# STEP 2 — DATA CLEANING
# ============================================================

train_clean = train_raw.copy()

# Fix monthly salaries: values < $10k that, multiplied by 12, fall in [10k, 500k]
mask_monthly = (
    (train_clean['annual.pay.usd'] < 10_000) &
    (train_clean['annual.pay.usd'] * 12 >= 10_000) &
    (train_clean['annual.pay.usd'] * 12 <= 500_000)
)
train_clean.loc[mask_monthly, 'annual.pay.usd'] *= 12
print(f"Monthly salaries corrected (×12): {mask_monthly.sum()}")

# Remove outliers outside [10k, 500k]
mask_valid = (
    (train_clean['annual.pay.usd'] >= 10_000) &
    (train_clean['annual.pay.usd'] <= 500_000)
)
n_removed = (~mask_valid).sum()
train_clean = train_clean[mask_valid].copy()
print(f"Removed observations (out of range): {n_removed}")
print(f"Remaining observations: {len(train_clean)}")

# Fix years inconsistencies
mask_inv = train_clean['coding.years.professional'] > train_clean['coding.years.total']
train_clean.loc[mask_inv, 'coding.years.professional'] = np.nan

mask_age = (train_clean['age.group'] == '18-24') & (train_clean['coding.years.total'] > 20)
train_clean.loc[mask_age, ['coding.years.total', 'coding.years.professional']] = np.nan

print(f"Fixed experience inconsistencies: {mask_inv.sum() + mask_age.sum()}")


In [ ]:
# ============================================================
# STEP 3 — STRATIFIED TRAIN/VAL SPLIT 80/20 (no data leakage)
# ============================================================

X_raw = train_clean.drop(columns=['annual.pay.usd'])
y_raw = train_clean['annual.pay.usd']
y_log = np.log1p(y_raw)

# Stratify on 5 quantile bins of log(salary)
y_bins = pd.qcut(y_log, q=5, labels=False)

X_train, X_val, y_train, y_val, y_train_log, y_val_log = train_test_split(
    X_raw, y_raw, y_log,
    test_size=0.2, random_state=42, stratify=y_bins
)

print(f"Train: {X_train.shape[0]} obs  |  Validation: {X_val.shape[0]} obs")


In [ ]:
# ============================================================
# STEP 4 — CONSTANTS FOR FEATURE ENGINEERING
# ============================================================

MULTI_COLS = [
    'prog.languages', 'databases', 'cloud.platforms', 'web.frameworks',
    'other.tech', 'dev.tools', 'dev.environments', 'personal.os',
    'work.os', 'project.mgmt.tools', 'comm.tools',
    'ai.search.tools', 'ai.tools.used', 'how.learned.coding', 'side.coding'
]

NOMINAL_COLS = [
    'region', 'employment.type', 'work.location', 'dev.role',
    'people.manager', 'industry', 'build.vs.buy', 'cloud.hosting',
    'first.help.source', 'uses.ai', 'ai.job.threat', 'is.dev.professional'
]

# Columns with >30% missing → create binary missing flag
HIGH_MISSING_COLS = [
    'cloud.hosting', 'daily.answer.time', 'daily.search.time',
    'first.help.source', 'industry', 'job.satisfaction',
    'experience.years', 'people.manager', 'other.tech',
    'ai.tools.used', 'ai.trust', 'ai.complex.rating', 'ai.sentiment',
]

ORDINAL_MAPS = {
    'age.group': {'18-24': 1, '25-34': 2, '35-44': 3, '45-54': 4, '55+': 5},
    'education': {
        'Primary/elementary school': 1,
        'Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)': 2,
        'Some college/university study without earning a degree': 3,
        'Associate degree (A.A., A.S., etc.)': 4,
        "Bachelor's degree (B.A., B.S., B.Eng., etc.)": 5,
        "Master's degree (M.A., M.S., M.Eng., MBA, etc.)": 6,
        'Professional degree (JD, MD, Ph.D, Ed.D, etc.)': 7,
        'Something else': 3,
    },
    'company.size': {
        'Just me - I am a freelancer, sole proprietor, etc.': 1,
        '2 to 9 employees': 2,         '10 to 19 employees': 3,
        '20 to 99 employees': 4,       '100 to 499 employees': 5,
        '500 to 999 employees': 6,     '1,000 to 4,999 employees': 7,
        '5,000 to 9,999 employees': 8, '10,000 or more employees': 9,
        "I don't know": np.nan,
    },
    'tech.purchase.influence': {
        'I have little or no influence': 1,
        'I have some influence': 2,
        'I have a great deal of influence': 3,
    },
    'ai.sentiment': {
        'Very unfavorable': 1, 'Unfavorable': 2, 'Indifferent': 3,
        'Unsure': 3, 'Favorable': 4, 'Very favorable': 5,
    },
    'ai.trust': {
        'Highly distrust': 1, 'Somewhat distrust': 2,
        'Neither trust nor distrust': 3,
        'Somewhat trust': 4, 'Highly trust': 5,
    },
    'ai.complex.rating': {
        'Very poor at handling complex tasks': 1,
        'Bad at handling complex tasks': 2,
        'Neither good or bad at handling complex tasks': 3,
        'Good, but not great at handling complex tasks': 4,
        'Very well at handling complex tasks': 5,
    },
    'daily.search.time': {
        'Less than 15 minutes a day': 1, '15-30 minutes a day': 2,
        '30-60 minutes a day': 3,        '60-120 minutes a day': 4,
        'Over 120 minutes a day': 5,
    },
    'daily.answer.time': {
        'Less than 15 minutes a day': 1, '15-30 minutes a day': 2,
        '30-60 minutes a day': 3,        '60-120 minutes a day': 4,
        'Over 120 minutes a day': 5,
    },
}

# Numeric map for company.size used in the manager_x_company interaction
COMPANY_MAP_INTER = {
    'Just me - I am a freelancer, sole proprietor, etc.': 1,
    '2 to 9 employees': 2,         '10 to 19 employees': 3,
    '20 to 99 employees': 4,       '100 to 499 employees': 5,
    '500 to 999 employees': 6,     '1,000 to 4,999 employees': 7,
    '5,000 to 9,999 employees': 8, '10,000 or more employees': 9,
    "I don't know": 4,
}

# Regions whose median salary > $50k → high-pay flag
TOP_REGIONS = ['R04', 'R02', 'R14', 'R11', 'R10', 'R18', 'R07', 'R08', 'R13']

HELP_SOURCE_RANK = {
    'AI-powered search (paid)': 5,
    'Slack search': 5,
    'Internal Developer portal': 4,
    'Do search of internal share drives/storage locations for documentation (i.e., not a structured knowledge base)': 4,
    'Traditional public search engine': 3,
    'A coworker': 2,
    'AI-powered search (free)': 2,
}

# Technologies associated with senior/high-pay profiles
SENIOR_TECHS = [
    'Terraform', 'Kubernetes', 'Amazon Web Services (AWS)',
    'Elasticsearch', 'Redis', 'BigQuery', 'Kafka', 'Ansible'
]
# Technologies associated with junior/low-pay profiles
JUNIOR_TECHS = ['MySQL', 'PHP', 'WordPress', 'jQuery', 'Bootstrap']

# Columns to scan for senior/junior tech counts
TECH_COLS_SENIOR = ['prog.languages', 'databases', 'cloud.platforms', 'web.frameworks', 'dev.tools']


In [ ]:
# ============================================================
# STEP 5 — HELPER FUNCTIONS
# ============================================================

def _count_techs(series, tech_list):
    """Count how many technologies from tech_list appear in each row of a semicolon-separated series."""
    return series.fillna('').apply(
        lambda x: sum(1 for t in tech_list if t in x)
    )


def exp_bucket(s):
    """Discretise professional experience into 5 ordered levels (0–4)."""
    s = s.fillna(0)
    return s.apply(lambda x: 0 if x <= 2 else
                              1 if x <= 5 else
                              2 if x <= 10 else
                              3 if x <= 15 else 4)


def process_multiselect(df_tr, df_te, col, top_n=15, y_tr_log=None):
    """
    For a semicolon-separated multi-select column:
    - Select top_n items by absolute correlation with log(target) (computed on train only).
    - Create binary indicator columns + a count column.
    """
    all_items = df_tr[col].dropna().str.split(';').explode().str.strip()
    all_items = all_items[all_items != '']
    unique_items = all_items.value_counts()
    unique_items = unique_items[unique_items >= 20].index.tolist()

    if y_tr_log is not None and len(unique_items) > 0:
        corr_dict = {}
        for item in unique_items:
            flag = df_tr[col].fillna('').apply(
                lambda x: int(item in [i.strip() for i in x.split(';')])
            )
            corr_dict[item] = abs(flag.corr(y_tr_log))
        top_items = sorted(corr_dict, key=corr_dict.get, reverse=True)[:top_n]
    else:
        top_items = all_items.value_counts().head(top_n).index.tolist()

    res_tr, res_te = {}, {}
    for item in top_items:
        safe = (item.replace(' ', '_').replace('/', '_').replace('.', '_')
                    .replace('+', 'plus').replace('(', '').replace(')', ''))
        cname = f'{col}__{safe}'
        res_tr[cname] = df_tr[col].fillna('').apply(
            lambda x: int(item in [i.strip() for i in x.split(';')])
        )
        res_te[cname] = df_te[col].fillna('').apply(
            lambda x: int(item in [i.strip() for i in x.split(';')])
        )

    count_col = f'{col}__count'
    res_tr[count_col] = df_tr[col].fillna('').apply(
        lambda x: len([i for i in x.split(';') if i.strip()])
    )
    res_te[count_col] = df_te[col].fillna('').apply(
        lambda x: len([i for i in x.split(';') if i.strip()])
    )
    return (pd.DataFrame(res_tr, index=df_tr.index),
            pd.DataFrame(res_te, index=df_te.index))


In [ ]:
def build_features(df_tr, df_te, y_tr=None):
    """
    Build the full feature matrix from raw data frames.
    All statistics (target encoding, top multiselect items) are derived
    exclusively from df_tr to prevent data leakage.
    """
    tr, te = df_tr.copy(), df_te.copy()
    parts_tr, parts_te = [], []

    # --- 5a. Binary missing flags for high-missingness columns ---
    for col in HIGH_MISSING_COLS:
        if col in tr.columns:
            parts_tr.append(tr[col].isnull().astype(int).rename(f'{col}__missing'))
            parts_te.append(te[col].isnull().astype(int).rename(f'{col}__missing'))

    # --- 5b. Ordinal encoding ---
    for col, mapping in ORDINAL_MAPS.items():
        if col in tr.columns:
            parts_tr.append(tr[col].map(mapping).rename(col))
            parts_te.append(te[col].map(mapping).rename(col))

    # --- 5c. Interaction: people manager × company size ---
    is_manager_tr = (tr['people.manager'] == 'People manager').astype(float)
    is_manager_te = (te['people.manager'] == 'People manager').astype(float)
    company_num_tr = tr['company.size'].map(COMPANY_MAP_INTER).fillna(4)
    company_num_te = te['company.size'].map(COMPANY_MAP_INTER).fillna(4)
    parts_tr.append((is_manager_tr * company_num_tr).rename('manager_x_company'))
    parts_te.append((is_manager_te * company_num_te).rename('manager_x_company'))

    # --- 5d. Numeric experience columns ---
    for col in ['coding.years.total', 'coding.years.professional',
                'experience.years', 'job.satisfaction']:
        if col in tr.columns:
            parts_tr.append(tr[col].rename(col))
            parts_te.append(te[col].rename(col))

    # Log-transform experience columns to reduce skewness
    for col in ['coding.years.total', 'coding.years.professional', 'experience.years']:
        if col in tr.columns:
            parts_tr.append(np.log1p(tr[col].fillna(0)).rename(f'{col}__log'))
            parts_te.append(np.log1p(te[col].fillna(0)).rename(f'{col}__log'))

    # Quadratic term for professional experience (captures diminishing returns)
    exp_pro_tr = tr['coding.years.professional'].fillna(0)
    exp_pro_te = te['coding.years.professional'].fillna(0)
    parts_tr.append((exp_pro_tr ** 2).rename('exp_professional_sq'))
    parts_te.append((exp_pro_te ** 2).rename('exp_professional_sq'))

    # --- 5e. Target encoding: region median log-salary (fit on train only) ---
    if y_tr is not None:
        region_median = (
            tr.assign(y=y_tr.values).groupby('region')['y'].median()
        )
        parts_tr.append(tr['region'].map(region_median).rename('region_target_enc'))
        parts_te.append(
            te['region'].map(region_median)
              .fillna(region_median.median())
              .rename('region_target_enc')
        )

    # Target encoding: dev.role and industry
    if y_tr is not None:
        for col_te in ['dev.role', 'industry']:
            if col_te in tr.columns:
                te_median = (
                    tr.assign(y=y_tr.values).groupby(col_te)['y'].median()
                )
                parts_tr.append(
                    tr[col_te].map(te_median)
                      .fillna(te_median.median())
                      .rename(f'{col_te}_target_enc')
                )
                parts_te.append(
                    te[col_te].map(te_median)
                      .fillna(te_median.median())
                      .rename(f'{col_te}_target_enc')
                )

    # --- 5f. One-Hot Encoding for nominal columns (fit on train only) ---
    nom = [c for c in NOMINAL_COLS if c in tr.columns]
    _ohe = OHE(sparse_output=False, drop='first', handle_unknown='ignore')
    ohe_tr_arr = _ohe.fit_transform(tr[nom].fillna('__MISSING__'))
    ohe_te_arr = _ohe.transform(te[nom].fillna('__MISSING__'))
    ohe_cols = _ohe.get_feature_names_out(nom)
    parts_tr.append(pd.DataFrame(ohe_tr_arr, index=tr.index, columns=ohe_cols))
    parts_te.append(pd.DataFrame(ohe_te_arr, index=te.index, columns=ohe_cols))

    # --- 5g. Multi-select columns: top-15 by correlation + count ---
    for col in MULTI_COLS:
        if col in tr.columns:
            p_tr, p_te = process_multiselect(tr, te, col, top_n=15, y_tr_log=y_tr)
            parts_tr.append(p_tr)
            parts_te.append(p_te)

    # --- 5h. Senior / junior tech stack counts ---
    total_senior_tr = sum(_count_techs(tr[c], SENIOR_TECHS) for c in TECH_COLS_SENIOR if c in tr.columns)
    total_senior_te = sum(_count_techs(te[c], SENIOR_TECHS) for c in TECH_COLS_SENIOR if c in te.columns)
    total_junior_tr = sum(_count_techs(tr[c], JUNIOR_TECHS) for c in TECH_COLS_SENIOR if c in tr.columns)
    total_junior_te = sum(_count_techs(te[c], JUNIOR_TECHS) for c in TECH_COLS_SENIOR if c in te.columns)
    parts_tr.append(pd.Series(total_senior_tr, index=tr.index, name='total_senior'))
    parts_te.append(pd.Series(total_senior_te, index=te.index, name='total_senior'))
    parts_tr.append(pd.Series(total_junior_tr, index=tr.index, name='total_junior'))
    parts_te.append(pd.Series(total_junior_te, index=te.index, name='total_junior'))
    senior_ratio_tr = total_senior_tr / (total_senior_tr + total_junior_tr + 0.1)
    senior_ratio_te = total_senior_te / (total_senior_te + total_junior_te + 0.1)
    parts_tr.append(pd.Series(senior_ratio_tr, index=tr.index, name='senior_ratio'))
    parts_te.append(pd.Series(senior_ratio_te, index=te.index, name='senior_ratio'))

    # --- 5i. Region-based binary flag (high-pay regions) ---
    is_top_tr = tr['region'].isin(TOP_REGIONS).astype(float)
    is_top_te = te['region'].isin(TOP_REGIONS).astype(float)
    parts_tr.append(is_top_tr.rename('is_top_region'))
    parts_te.append(is_top_te.rename('is_top_region'))

    # --- 5j. Experience bucket (ordinal discretisation 0–4) ---
    parts_tr.append(exp_bucket(tr['coding.years.professional']).rename('exp_bucket'))
    parts_te.append(exp_bucket(te['coding.years.professional']).rename('exp_bucket'))

    # --- 5k. Interaction features ---
    is_senior_tr = (tr['coding.years.professional'].fillna(0) >= 10).astype(float)
    is_senior_te = (te['coding.years.professional'].fillna(0) >= 10).astype(float)
    is_junior_tr = (tr['coding.years.professional'].fillna(0) <= 2).astype(float)
    is_junior_te = (te['coding.years.professional'].fillna(0) <= 2).astype(float)
    is_remote_tr = (tr['work.location'] == 'Remote').astype(float)
    is_remote_te = (te['work.location'] == 'Remote').astype(float)
    is_freelance_tr = tr['employment.type'].str.contains('Freelance|Self', case=False, na=False).astype(float)
    is_freelance_te = te['employment.type'].str.contains('Freelance|Self', case=False, na=False).astype(float)

    # senior developer in a high-pay region
    parts_tr.append((is_senior_tr * is_top_tr).rename('senior_x_top_region'))
    parts_te.append((is_senior_te * is_top_te).rename('senior_x_top_region'))

    # junior developer in a low-pay region
    parts_tr.append((is_junior_tr * (1 - is_top_tr)).rename('junior_x_low_region'))
    parts_te.append((is_junior_te * (1 - is_top_te)).rename('junior_x_low_region'))

    # remote + senior + high-pay region (triple interaction)
    parts_tr.append((is_remote_tr * is_senior_tr * is_top_tr).rename('remote_senior_top'))
    parts_te.append((is_remote_te * is_senior_te * is_top_te).rename('remote_senior_top'))

    # freelance senior
    parts_tr.append((is_freelance_tr * is_senior_tr).rename('freelance_senior'))
    parts_te.append((is_freelance_te * is_senior_te).rename('freelance_senior'))

    # --- 5l. Side-coding binary flags ---
    side_tr = tr['side.coding'].fillna('')
    side_te = te['side.coding'].fillna('')

    parts_tr.append(side_tr.apply(lambda x: int('open-source' in x.lower())).rename('side_opensource'))
    parts_te.append(side_te.apply(lambda x: int('open-source' in x.lower())).rename('side_opensource'))

    parts_tr.append(side_tr.apply(lambda x: int('Bootstrapping' in x)).rename('side_bootstrap'))
    parts_te.append(side_te.apply(lambda x: int('Bootstrapping' in x)).rename('side_bootstrap'))

    parts_tr.append(side_tr.apply(lambda x: int('School' in x or 'academic' in x.lower())).rename('side_student'))
    parts_te.append(side_te.apply(lambda x: int('School' in x or 'academic' in x.lower())).rename('side_student'))

    # student in a low-pay region
    side_student_tr = side_tr.apply(lambda x: int('School' in x or 'academic' in x.lower()))
    side_student_te = side_te.apply(lambda x: int('School' in x or 'academic' in x.lower()))
    parts_tr.append((side_student_tr * (1 - is_top_tr)).rename('student_x_low_region'))
    parts_te.append((side_student_te * (1 - is_top_te)).rename('student_x_low_region'))

    # --- 5m. First help source: ordinal autonomy rank ---
    parts_tr.append(tr['first.help.source'].map(HELP_SOURCE_RANK).fillna(3).rename('help_source_rank'))
    parts_te.append(te['first.help.source'].map(HELP_SOURCE_RANK).fillna(3).rename('help_source_rank'))

    # --- 5n. Age squared (captures salary peak at 35–44 then decline) ---
    age_num_tr = tr['age.group'].map({'18-24': 1, '25-34': 2, '35-44': 3, '45-54': 4, '55+': 5}).fillna(2.5)
    age_num_te = te['age.group'].map({'18-24': 1, '25-34': 2, '35-44': 3, '45-54': 4, '55+': 5}).fillna(2.5)
    parts_tr.append((age_num_tr ** 2).rename('age_squared'))
    parts_te.append((age_num_te ** 2).rename('age_squared'))

    # --- 5o. Young but experienced (age 25-34 AND professional exp >= 7 years) ---
    is_young_senior_tr = (
        tr['age.group'].isin(['25-34']) &
        (tr['coding.years.professional'].fillna(0) >= 7)
    ).astype(float)
    is_young_senior_te = (
        te['age.group'].isin(['25-34']) &
        (te['coding.years.professional'].fillna(0) >= 7)
    ).astype(float)
    parts_tr.append(is_young_senior_tr.rename('is_young_senior'))
    parts_te.append(is_young_senior_te.rename('is_young_senior'))

    # --- Assemble final matrices ---
    X_tr = pd.concat(parts_tr, axis=1).astype(float)
    X_te = pd.concat(parts_te, axis=1).astype(float)
    X_te = X_te.reindex(columns=X_tr.columns, fill_value=0)
    return X_tr, X_te


In [ ]:
# ============================================================
# STEP 6 — FEATURE ENGINEERING ON TRAIN/VAL SPLIT
# ============================================================

X_tr_fe, X_val_fe  = build_features(X_train, X_val,     y_tr=y_train_log)
_,        X_test_fe = build_features(X_train, test_raw,  y_tr=y_train_log)

# Impute residual NaNs with median — fit ONLY on X_train
imputer    = SimpleImputer(strategy='median')
X_tr_imp   = pd.DataFrame(imputer.fit_transform(X_tr_fe),  columns=X_tr_fe.columns)
X_val_imp  = pd.DataFrame(imputer.transform(X_val_fe),     columns=X_val_fe.columns)
X_test_imp = pd.DataFrame(imputer.transform(X_test_fe),    columns=X_test_fe.columns)

print(f"Total features generated: {X_tr_imp.shape[1]}")
assert X_tr_imp.isnull().sum().sum() == 0,  "NaNs remaining in X_train!"
assert X_val_imp.isnull().sum().sum() == 0, "NaNs remaining in X_val!"


In [ ]:
# ============================================================
# STEP 7 — STATISTICAL FEATURE SELECTION
# Pearson |corr| < 0.01 for continuous → remove
# ANOVA p >= 0.05 for binary → remove
# Manual redundant removal
# Protected features are never removed
# ============================================================

n_unique      = X_tr_imp.nunique()
is_binary     = n_unique <= 2
is_continuous = n_unique > 2

# Pearson correlation with log(salary) — continuous variables
corr_abs = X_tr_imp.corrwith(y_train_log).abs()
remove_low_corr = corr_abs[is_continuous & (corr_abs < 0.01)].index.tolist()

# ANOVA F-statistic — binary variables
f_stats, p_values = f_regression(X_tr_imp, y_train_log)
anova_df = pd.DataFrame({'p_value': p_values}, index=X_tr_imp.columns)
remove_non_sig = anova_df[is_binary & (anova_df['p_value'] >= 0.05)].index.tolist()

remove_by_stats = list(set(remove_low_corr) | set(remove_non_sig))

# Manual redundant features (verified via cross-correlation analysis)
remove_redundant = [
    'coding.years.total',
    'coding.years.total__log',
    'age.group',
    'is_top_region',
    'total_junior',
    'senior_ratio',
]

# Protected features — never removed regardless of statistics
PROTECTED_FEATURES = [
    'exp_bucket', 'region_target_enc', 'coding.years.professional',
    'coding.years.professional__log', 'exp_professional_sq',
    'is_young_senior', 'total_senior', 'senior_x_top_region',
    'remote_senior_top', 'junior_x_low_region', 'freelance_senior',
    'company.size', 'tech.purchase.influence',
]

remove_all   = (set(remove_by_stats) | set(remove_redundant)) - set(PROTECTED_FEATURES)
cols_to_remove = [c for c in remove_all if c in X_tr_imp.columns]

X_tr_sel   = X_tr_imp.drop(columns=cols_to_remove)
X_val_sel  = X_val_imp.drop(columns=cols_to_remove)
X_test_sel = X_test_imp.drop(columns=cols_to_remove)

print(f"Features before selection: {X_tr_imp.shape[1]}")
print(f"Features removed:          {len(cols_to_remove)}")
print(f"Features after selection:  {X_tr_sel.shape[1]}")


In [ ]:
# ============================================================
# STEP 8 — VALIDATION: SVR RBF WITH BEST HYPERPARAMETERS
# Pipeline: VarianceThreshold -> Yeo-Johnson -> StandardScaler -> SVR
# ============================================================

def rmse_usd(y_true, y_pred_log):
    """Compute RMSE in USD by inverting the log1p transformation."""
    return np.sqrt(mean_squared_error(np.array(y_true), np.expm1(y_pred_log)))

pipe_svr = Pipeline([
    ('vt',    VarianceThreshold(threshold=0.01)),
    ('yj',    PowerTransformer(method='yeo-johnson')),
    ('sc',    StandardScaler()),
    ('model', SVR(kernel='rbf', C=0.5, epsilon=0.05, gamma=0.0005,
                  max_iter=50_000, tol=0.01)),
])

pipe_svr.fit(X_tr_sel, y_train_log)
val_pred_log = pipe_svr.predict(X_val_sel)
val_rmse     = rmse_usd(y_val, val_pred_log)

print(f"\nValidation RMSE (USD): ${val_rmse:,.0f}")


In [ ]:
# ============================================================
# STEP 9 — RETRAIN ON FULL TRAINING SET
# Target encoding is re-derived on the full clean training set,
# but using medians from the 80% split to avoid leakage
# ============================================================

X_all_raw = train_clean.drop(columns=['annual.pay.usd'])
y_all_log  = np.log1p(train_clean['annual.pay.usd'])

# Re-derive region/role/industry medians from X_train (80% split) only
region_median_train = (
    X_train.assign(y=y_train_log.values).groupby('region')['y'].median()
)

X_all_fe, X_test_fe2 = build_features(X_all_raw, test_raw, y_tr=y_all_log)

# Overwrite target encodings with medians from X_train to prevent leakage
X_all_fe['region_target_enc'] = (
    X_all_raw['region'].map(region_median_train).fillna(region_median_train.median())
)
X_test_fe2['region_target_enc'] = (
    test_raw['region'].map(region_median_train).fillna(region_median_train.median())
)

for col_te in ['dev.role', 'industry']:
    te_median_train = (
        X_train.assign(y=y_train_log.values).groupby(col_te)['y'].median()
    )
    X_all_fe[f'{col_te}_target_enc'] = (
        X_all_raw[col_te].map(te_median_train).fillna(te_median_train.median())
    )
    X_test_fe2[f'{col_te}_target_enc'] = (
        test_raw[col_te].map(te_median_train).fillna(te_median_train.median())
    )

# Impute on the full training set
imputer_full = SimpleImputer(strategy='median')
X_all_imp    = pd.DataFrame(imputer_full.fit_transform(X_all_fe),  columns=X_all_fe.columns)
X_test_imp2  = pd.DataFrame(imputer_full.transform(X_test_fe2),    columns=X_test_fe2.columns)

# Apply the same feature selection mask
cols_rm_final = [c for c in cols_to_remove if c in X_all_imp.columns]
X_all_final   = X_all_imp.drop(columns=cols_rm_final)
X_test_final  = X_test_imp2.drop(columns=cols_rm_final)

print(f"\nFull train: {X_all_final.shape}  |  Kaggle test: {X_test_final.shape}")

# Retrain the final model on the complete training data
final_model = Pipeline([
    ('vt',    VarianceThreshold(threshold=0.01)),
    ('yj',    PowerTransformer(method='yeo-johnson')),
    ('sc',    StandardScaler()),
    ('model', SVR(kernel='rbf', C=0.5, epsilon=0.05, gamma=0.0005,
                  max_iter=50_000, tol=0.01)),
])

final_model.fit(X_all_final, y_all_log)
print("Retrain on full dataset completed.")


In [ ]:
# ============================================================
# STEP 10 — GENERATE PREDICTIONS AND SAVE submission.csv
# ============================================================

y_pred_log = final_model.predict(X_test_final)
y_pred_usd = np.expm1(y_pred_log).clip(min=0)

print(f"\nTest predictions summary:")
print(f"  Min:    ${y_pred_usd.min():>10,.0f}")
print(f"  Median: ${np.median(y_pred_usd):>10,.0f}")
print(f"  Max:    ${y_pred_usd.max():>10,.0f}")

submission = pd.DataFrame({
    'id':             test_raw['id'],
    'annual.pay.usd': y_pred_usd,
})

submission.to_csv('submission.csv', index=False)
print(f"\nFile saved: submission.csv  ({len(submission)} rows)")
print("Upload this file to Kaggle!")
